# 06 — Dashboard Interactif Dash

**Objectif :** Prototypage et documentation du dashboard interactif.  
**Application complète :** `dashboard/app.py` → `python app.py` → http://localhost:8050

---

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from utils import load_dataframe, GRAVITE_COLORS

df = load_dataframe('../data/processed/accidents_clean.parquet')
print(f'Dataset : {df.shape}')

2026-05-21 19:32:42 | INFO     | utils | DataFrame chargé ← ..\data\processed\accidents_clean.parquet (49,783 lignes)


Dataset : (49783, 30)


## 1. Architecture du Dashboard

```
dashboard/
├── app.py               ← Application principale Dash
├── assets/
│   └── custom.css       ← Styles CSS personnalisés
└── components/
    ├── kpi_cards.py     ← Composants KPI
    └── charts.py        ← Fonctions graphiques
```

### Onglets du dashboard :
1. **Vue d'ensemble** : KPIs, distribution gravité, tendances
2. **Analyse temporelle** : Heatmap, profil horaire, évolution annuelle
3. **Cartographie** : Scatter map + table des villes
4. **Causes & Risques** : Causes, EPI, météo
5. **Économique** : Coûts par région/cause
6. **Prédiction ML** : Formulaire de prédiction interactif

## 2. Prototypage des graphiques clés

In [2]:
# ── KPI Cards ────────────────────────────────────────────────────────────
total_accidents = len(df)
total_deces     = int(df['nombre_deces'].sum())
tx_mortalite    = total_deces / total_accidents * 100
cout_total_G    = df['cout_estime'].sum() / 1e9
nb_regions      = df['region'].nunique()

kpis = {
    'Total Accidents':    f"{total_accidents:,}",
    'Décès Totaux':       f"{total_deces:,}",
    'Taux Mortalité':     f"{tx_mortalite:.2f}%",
    'Coût Total (GFCFA)': f"{cout_total_G:.2f}",
    'Régions':            str(nb_regions),
}

for key, val in kpis.items():
    print(f'  {key:<25} : {val}')

  Total Accidents           : 49,783
  Décès Totaux              : 21,112
  Taux Mortalité            : 42.41%
  Coût Total (GFCFA)        : 106.86
  Régions                   : 5


In [3]:
# ── Graphique indicateur style gauge ─────────────────────────────────────
fig_gauges = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]]
)

fig_gauges.add_trace(go.Indicator(
    mode='number+delta',
    value=total_accidents,
    title={'text': 'Total Accidents'},
    number={'font': {'color': '#3498db', 'size': 40}},
), row=1, col=1)

fig_gauges.add_trace(go.Indicator(
    mode='number+gauge',
    value=tx_mortalite,
    title={'text': 'Taux Mortalité (%)'},
    number={'suffix': '%', 'font': {'color': '#e74c3c', 'size': 40}},
    gauge={
        'axis': {'range': [0, 20]},
        'bar': {'color': '#e74c3c'},
        'steps': [
            {'range': [0, 5], 'color': '#eafaf1'},
            {'range': [5, 10], 'color': '#fef9e7'},
            {'range': [10, 20], 'color': '#fdedec'},
        ]
    }
), row=1, col=2)

fig_gauges.add_trace(go.Indicator(
    mode='number',
    value=cout_total_G,
    title={'text': 'Coût Total (GFCFA)'},
    number={'suffix': ' G', 'font': {'color': '#f39c12', 'size': 40}},
), row=1, col=3)

fig_gauges.update_layout(height=220, margin=dict(t=20, b=10))
fig_gauges.show()

In [4]:
# ── Treemap accidents par région × gravité ────────────────────────────────
treemap_data = (
    df.groupby(['region', 'gravite'])
    .size().reset_index(name='count')
)

treemap_data['gravite'] = treemap_data['gravite'].astype(str)
treemap_data['region'] = treemap_data['region'].astype(str)

fig_treemap = px.treemap(
    treemap_data,
    path=[px.Constant('Togo'), 'region', 'gravite'],
    values='count',
    color='gravite',
    color_discrete_map=GRAVITE_COLORS,
    title='Treemap : Accidents par Région et Gravité'
)
fig_treemap.update_layout(height=450)
fig_treemap.show()

In [5]:
# ── Scatter 3D : Vitesse × Heure × Gravité ───────────────────────────────
sample_3d = df.sample(3000, random_state=42)

fig_3d = px.scatter_3d(
    sample_3d,
    x='heure', y='vitesse_estimee', z='intervention_secours',
    color='gravite', color_discrete_map=GRAVITE_COLORS,
    symbol='type_route',
    opacity=0.6,
    title='Vue 3D : Heure × Vitesse × Délai Intervention (couleur = Gravité)',
    labels={
        'heure': 'Heure', 'vitesse_estimee': 'Vitesse (km/h)',
        'intervention_secours': 'Délai intervention (min)'
    }
)
fig_3d.update_layout(height=550, scene=dict(
    xaxis_title='Heure',
    yaxis_title='Vitesse (km/h)',
    zaxis_title='Délai intervention (min)'
))
fig_3d.show()

In [6]:
# ── Parallel coordinates : profil des accidents mortels ──────────────────
mortels_sample = df[df['gravite'] == 'Mortel'].sample(min(1000, len(df[df['gravite']=='Mortel'])), random_state=42)
mortels_sample['gravite_num'] = mortels_sample['gravite_num'].astype(float)

fig_parallel = px.parallel_coordinates(
    mortels_sample,
    color='gravite_num',
    dimensions=['heure', 'vitesse_estimee', 'limitation_vitesse',
                'nombre_vehicules', 'nombre_victimes', 'intervention_secours'],
    color_continuous_scale=px.colors.sequential.YlOrRd,
    title='Coordonnées Parallèles — Profil des Accidents Mortels'
)
fig_parallel.update_layout(height=450)
fig_parallel.show()

In [7]:
# ── Violin plot multidimensionnel ─────────────────────────────────────────
fig_violin = px.violin(
    df, y='cout_estime', x='gravite', color='type_route',
    category_orders={'gravite': ['Léger', 'Grave', 'Mortel']},
    box=True, points=False,
    title='Distribution du Coût par Gravité et Type de Route',
    labels={'cout_estime': 'Coût estimé (FCFA)', 'gravite': 'Gravité'}
)
fig_violin.update_layout(height=450)
fig_violin.show()

## 3. Instructions de lancement du Dashboard

In [8]:
print('''
═══════════════════════════════════════════════════
LANCEMENT DU DASHBOARD
═══════════════════════════════════════════════════

Option 1 : Terminal
   cd togo-road-accidents-capstone/dashboard
   python app.py
   → http://localhost:8050

Option 2 : Depuis ce notebook
   (Décommentez la cellule suivante)

Fonctionnalités :
   ✓ Filtres dynamiques (région, année, gravité, véhicule)
   ✓ 6 onglets d'analyse
   ✓ 15+ graphiques interactifs
   ✓ Carte des accidents
   ✓ Prédicteur ML intégré
   ✓ Export de données
═══════════════════════════════════════════════════
''')


═══════════════════════════════════════════════════
LANCEMENT DU DASHBOARD
═══════════════════════════════════════════════════

Option 1 : Terminal
   cd togo-road-accidents-capstone/dashboard
   python app.py
   → http://localhost:8050

Option 2 : Depuis ce notebook
   (Décommentez la cellule suivante)

Fonctionnalités :
   ✓ Filtres dynamiques (région, année, gravité, véhicule)
   ✓ 6 onglets d'analyse
   ✓ 15+ graphiques interactifs
   ✓ Carte des accidents
   ✓ Prédicteur ML intégré
   ✓ Export de données
═══════════════════════════════════════════════════



In [9]:
# # Lancement du dashboard depuis le notebook (décommenter)
# import subprocess
# import os
# os.chdir('../dashboard')
# subprocess.Popen(['python', 'app.py'])
# print('Dashboard démarré → http://localhost:8050')